# Hessian — companion notebook

Companion for [`hessian.md`](./hessian.md). We compare gradient descent vs Newton's method on the Rosenbrock function (ill-conditioned, classic test case).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

## Rosenbrock function

$f(x, y) = (1 - x)^2 + 100(y - x^2)^2$. Minimum at $(1, 1)$. Notoriously curved valley — pathological for gradient descent.

In [ ]:
def f(p):
    x, y = p
    return (1 - x)**2 + 100 * (y - x**2)**2
def grad(p):
    x, y = p
    return np.array([-2*(1-x) - 400*x*(y - x**2), 200*(y - x**2)])
def hess(p):
    x, y = p
    return np.array([[2 - 400*(y - x**2) + 800*x**2, -400*x],[-400*x, 200]])

def gd(x0, lr=1e-3, steps=2000):
    path = [np.array(x0)]
    for _ in range(steps):
        path.append(path[-1] - lr*grad(path[-1]))
    return np.array(path)

def newton(x0, steps=20):
    path = [np.array(x0)]
    for _ in range(steps):
        H = hess(path[-1])
        # damped solve to handle indefiniteness near saddle-like region
        H_reg = H + 1e-6 * np.eye(2)
        step = np.linalg.solve(H_reg, grad(path[-1]))
        path.append(path[-1] - step)
    return np.array(path)

p0 = np.array([-1.2, 1.0])
gd_path = gd(p0)
nw_path = newton(p0)
print(f'GD final loss   after 2000 steps: {f(gd_path[-1]):.6g}, position: {gd_path[-1]}')
print(f'Newton final loss after  20 steps: {f(nw_path[-1]):.6g}, position: {nw_path[-1]}')

In [ ]:
xs = np.linspace(-1.6, 1.6, 400); ys = np.linspace(-0.5, 2.0, 400)
X, Y = np.meshgrid(xs, ys); Z = (1-X)**2 + 100*(Y - X**2)**2
fig, ax = plt.subplots(figsize=(8, 5))
ax.contour(X, Y, np.log10(Z + 1), levels=20, cmap='Greys', alpha=0.6)
ax.plot(gd_path[::20, 0], gd_path[::20, 1], '.-', label='Gradient descent', alpha=0.5)
ax.plot(nw_path[:, 0], nw_path[:, 1], 'o-', label='Newton', color='tab:red')
ax.scatter([1],[1], color='gold', s=120, marker='*', label='minimum (1,1)', zorder=5)
ax.set_aspect('equal'); ax.legend(); ax.set_title('Newton vs GD on Rosenbrock')
plt.show()